# The 0-π qubit

The 0-π qubit is a floating ring of two junctions and two inductors, cross-coupled by two capacitors [Brooks, Kitaev, and Preskill, Phys. Rev. A **87**, 052306 (2013)](https://doi.org/10.1103/PhysRevA.87.052306).
Its three dynamical modes are the compact $\theta$, the extended $\phi$, and the harmonic $\zeta$; this example takes the symmetric circuit.

In [ ]:
import sympy as sym
import circuit_quantization as cq
from circuit_quantization import Branch, Circuit, Loops, quantize

## Circuit

No node is grounded, so the circuit is floating; the package removes the free center-of-charge mode.

In [ ]:
C_J, C, L, E_J = sym.symbols("C_J C L E_J", positive=True)
varphi_ext = sym.Symbol("varphi_ext", real=True)

zero_pi = Circuit([
    Branch("J1", 1, 2, C=C_J, EJ=E_J),
    Branch("J2", 3, 4, C=C_J, EJ=E_J),
    Branch("L1", 2, 3, L=L),
    Branch("L2", 4, 1, L=L),
    Branch("C1", 1, 3, C=C),
    Branch("C2", 2, 4, C=C),
])
cq.draw_circuit_diagram(zero_pi, layout={1: (0, 1), 2: (1, 1), 3: (1, 0), 4: (0, 0)})

## Loops

The superconducting ring carries the external flux.
The two loops closed by the capacitors are capacitively broken and carry none.

In [ ]:
loops = Loops.from_branch_dict(
    zero_pi,
    {"ring": ("J1", "L1", "J2", "L2"), "c1": ("C1", "J1", "L1"), "c2": ("C2", "J2", "L1")},
    {"ring": cq.phase_to_weber(varphi_ext), "c1": 0, "c2": 0},
)

## Quantization

In [ ]:
result = quantize(zero_pi, loops, simplify=True)
assert result.coords.types == ("c", "e", "h")
result

The compact mode (`c`) and the extended mode (`e`) appear together in both junction cosines.
The harmonic mode (`h`) enters only the inductive and charging energies.
`result.coords.Z` gives each retained coordinate in terms of the node phases:

In [ ]:
result.coords.Z